# 🎙️ Clone your voice for your lessons — Colab

Speaks your lessons in **your voice**. The GitHub repo is used ONLY for the
voice-engine code — **your lesson content is NOT on GitHub**; you upload it here.

## 5 steps:
1. **Runtime → Change runtime type → GPU → Save**
2. **Cell 1** — setup (~5 min first time)
3. **Cell 2** — upload your **voice clip**: always `reference_30s.mp3`
4. **Cell 3** — upload your **lesson script file(s)** (the `.json` files Claude sends you; select several at once for batch)
5. **Cell 4** clones them all → **Cell 5** downloads `clones.zip`

Send `clones.zip` to Claude — he builds the final videos.

In [ ]:
# CELL 1 — Setup: voice-engine code + install (no lessons here). ~5 min first time.
!nvidia-smi -L || print('NO GPU: Runtime -> Change runtime type -> GPU, then re-run.')
import os
USER, REPO, BRANCH = 'PrashantRMarathe', 'voice-replacer', 'add-course-video'
if os.path.isdir(REPO):
    !cd {REPO} && git fetch -q && git checkout -q {BRANCH} && git pull -q
else:
    !git clone -q -b {BRANCH} https://github.com/{USER}/{REPO}.git
%cd {REPO}
!pip install -q coqui-tts 'transformers<5' ffmpeg-python soundfile num2words
!pip install -q f5-tts 2>/dev/null || print('(f5-tts optional, skipped)')
import importlib.metadata as md
try:
    print('F5-TTS version:', md.version('f5-tts'), '(tell Claude if the voice ever sounds different)')
except Exception:
    print('F5-TTS not installed')
print('\n✅ Setup done. Run Cell 2.')

In [ ]:
# CELL 2 — Upload your VOICE clip: ALWAYS use reference_30s.mp3
# (your original voice clip, the one behind the good Oct 19 voice). The same
# clip every time keeps every lesson in the same voice and loudness.
from google.colab import files
print('Choose your VOICE clip (mp3/wav)...')
reference_path = list(files.upload().keys())[0]
print('\n✅ Voice:', reference_path, '— run Cell 3.')

In [ ]:
# CELL 3 — Upload your LESSON script file(s): the .json files Claude sends you.
# You can select MANY at once (one per lesson) for a batch run.
import os
from google.colab import files
os.makedirs('my_lessons', exist_ok=True)
print('Choose your lesson .json file(s)...')
up = files.upload()
for fn, data in up.items():
    open(f'my_lessons/{fn}', 'wb').write(data)
print('\n✅ Lessons uploaded:', list(up.keys()), '— run Cell 4.')

In [ ]:
# CELL 4 — Clone your voice for every uploaded lesson (model loads once)
import os, glob, json, time
os.environ['COQUI_TOS_AGREED'] = '1'
import config, tts
config.TTS_BACKEND = 'f5'   # natural English; use 'xtts' for other languages
config.LANGUAGE = 'en'

files_in = sorted(glob.glob('my_lessons/*.json'))
print('Lessons to clone:', [os.path.splitext(os.path.basename(f))[0] for f in files_in])
backend = tts.get_backend()   # load the voice model ONCE for all lessons

t0 = time.time()
for fp in files_in:
    name = os.path.splitext(os.path.basename(fp))[0]
    out_dir = f'clones/{name}'; os.makedirs(out_dir, exist_ok=True)
    scripts = json.load(open(fp, encoding='utf-8'))
    print(f'\n=== {name}: {len(scripts)} scenes ===')
    for i, (key, text) in enumerate(scripts.items(), 1):
        out = f'{out_dir}/{key}.wav'
        if os.path.exists(out):
            print(f'  [{i}/{len(scripts)}] skip {key}'); continue
        backend.synth_to_file(text=text, speaker_wav=reference_path,
                              language=config.LANGUAGE, out_path=out)
        print(f'  [{i}/{len(scripts)}] {key}  ({time.time()-t0:.0f}s)')
print('\n✅ Done. Run Cell 5.')

In [ ]:
# CELL 5 — Download everything (one zip, a folder per lesson)
!zip -rq clones.zip clones && echo 'zipped clones.zip'
from google.colab import files
files.download('clones.zip')